In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

print("Libraries imported!")

Libraries imported!


In [2]:
data = [
    {
        "sanskrit": "कर्मण्येवाधिकारस्ते मा फलेषु कदाचन",
        "english": "You have the right to perform your duty, but not to the fruits of action."
    },
    {
        "sanskrit": "यदा यदा हि धर्मस्य ग्लानिर्भवति भारत",
        "english": "Whenever righteousness declines, I manifest myself."
    },
    {
        "sanskrit": "योगस्थः कुरु कर्माणि",
        "english": "Perform action while established in yoga."
    },
    {
        "sanskrit": "न हि ज्ञानेन सदृशं पवित्रमिह विद्यते",
        "english": "Nothing is as purifying as knowledge."
    },
    {
        "sanskrit": "उद्धरेदात्मनात्मानं नात्मानमवसादयेत्",
        "english": "One should elevate oneself by one’s own mind and not degrade oneself."
    }
]

df = pd.DataFrame(data)
df

,sanskrit,english
0,कर्मण्येवाधिकारस्ते मा फलेषु कदाचन,"You have the right to perform your duty, but n..."
1,यदा यदा हि धर्मस्य ग्लानिर्भवति भारत,"Whenever righteousness declines, I manifest my..."
2,योगस्थः कुरु कर्माणि,Perform action while established in yoga.
3,न हि ज्ञानेन सदृशं पवित्रमिह विद्यते,Nothing is as purifying as knowledge.
4,उद्धरेदात्मनात्मानं नात्मानमवसादयेत्,One should elevate oneself by one’s own mind a...


In [3]:
model = SentenceTransformer("intfloat/multilingual-e5-small")

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model loaded successfully!


In [4]:
train_examples = []

for row in data:
    train_examples.append(
        InputExample(texts=[row["english"], row["sanskrit"]])
    )

print("Training examples:", len(train_examples))

Training examples: 5


In [5]:
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=2
)

loss = MultipleNegativesRankingLoss(model)

print("Dataloader ready!")

Dataloader ready!


In [6]:
model.fit(
    train_objectives=[(train_dataloader, loss)],
    epochs=3,
    warmup_steps=2,
    show_progress_bar=True
)

print("Fine-tuning completed!")

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Fine-tuning completed!


In [7]:
model.save("sanskrit-retrieval-model")

print("Model saved successfully!")

Model saved successfully!


In [8]:
documents = df["sanskrit"].tolist()

doc_embeddings = model.encode(
    documents,
    normalize_embeddings=True
)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(np.array(doc_embeddings, dtype="float32"))

print("FAISS index created!")

FAISS index created!


In [9]:
query = "What does the Gita teach about performing duty without attachment?"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
)

scores, ids = index.search(
    np.array(query_embedding, dtype="float32"),
    k=3
)

print("Query:", query)
print("\nTop Results:\n")

for rank, idx in enumerate(ids[0], 1):
    print(f"Rank {rank}:")
    print("Sanskrit:", documents[idx])
    print("English:", df.iloc[idx]["english"])
    print("Score:", round(float(scores[0][rank-1]), 4))
    print("-" * 50)

Query: What does the Gita teach about performing duty without attachment?

Top Results:

Rank 1:
Sanskrit: न हि ज्ञानेन सदृशं पवित्रमिह विद्यते
English: Nothing is as purifying as knowledge.
Score: 0.7717
--------------------------------------------------
Rank 2:
Sanskrit: योगस्थः कुरु कर्माणि
English: Perform action while established in yoga.
Score: 0.7694
--------------------------------------------------
Rank 3:
Sanskrit: कर्मण्येवाधिकारस्ते मा फलेषु कदाचन
English: You have the right to perform your duty, but not to the fruits of action.
Score: 0.7669
--------------------------------------------------


In [10]:
def search_sanskrit(query, top_k=3):
    query_embedding = model.encode([query], normalize_embeddings=True)

    scores, ids = index.search(
        np.array(query_embedding, dtype="float32"),
        k=top_k
    )

    results = []
    for idx, score in zip(ids[0], scores[0]):
        results.append({
            "sanskrit": documents[idx],
            "english": df.iloc[idx]["english"],
            "score": round(float(score), 4)
        })

    return results

print("Search function created!")

Search function created!


In [11]:
queries = [
    "teaching about karma",
    "importance of knowledge",
    "how to improve oneself",
    "action while in yoga"
]

for q in queries:
    print(f"\nQuery: {q}")
    print("Best Match:")

    result = search_sanskrit(q, top_k=1)[0]

    print("Sanskrit:", result["sanskrit"])
    print("English:", result["english"])
    print("Score:", result["score"])
    print("=" * 60)


Query: teaching about karma
Best Match:
Sanskrit: योगस्थः कुरु कर्माणि
English: Perform action while established in yoga.
Score: 0.8355

Query: importance of knowledge
Best Match:
Sanskrit: न हि ज्ञानेन सदृशं पवित्रमिह विद्यते
English: Nothing is as purifying as knowledge.
Score: 0.8292

Query: how to improve oneself
Best Match:
Sanskrit: उद्धरेदात्मनात्मानं नात्मानमवसादयेत्
English: One should elevate oneself by one’s own mind and not degrade oneself.
Score: 0.8153

Query: action while in yoga
Best Match:
Sanskrit: योगस्थः कुरु कर्माणि
English: Perform action while established in yoga.
Score: 0.8948


In [12]:
test_cases = [
    {
        "query": "perform duty without attachment",
        "expected": "कर्मण्येवाधिकारस्ते मा फलेषु कदाचन"
    },
    {
        "query": "nothing is as purifying as knowledge",
        "expected": "न हि ज्ञानेन सदृशं पवित्रमिह विद्यते"
    },
    {
        "query": "uplift yourself by your own mind",
        "expected": "उद्धरेदात्मनात्मानं नात्मानमवसादयेत्"
    }
]

correct = 0

for test in test_cases:
    prediction = search_sanskrit(test["query"], top_k=1)[0]["sanskrit"]

    if prediction == test["expected"]:
        correct += 1

    print(f"Query: {test['query']}")
    print(f"Expected: {test['expected']}")
    print(f"Predicted: {prediction}")
    print()

accuracy = correct / len(test_cases)
print(f"Recall@1: {accuracy:.2f}")

Query: perform duty without attachment
Expected: कर्मण्येवाधिकारस्ते मा फलेषु कदाचन
Predicted: कर्मण्येवाधिकारस्ते मा फलेषु कदाचन

Query: nothing is as purifying as knowledge
Expected: न हि ज्ञानेन सदृशं पवित्रमिह विद्यते
Predicted: न हि ज्ञानेन सदृशं पवित्रमिह विद्यते

Query: uplift yourself by your own mind
Expected: उद्धरेदात्मनात्मानं नात्मानमवसादयेत्
Predicted: उद्धरेदात्मनात्मानं नात्मानमवसादयेत्

Recall@1: 1.00


In [13]:
results = []

for test in test_cases:
    prediction = search_sanskrit(test["query"], top_k=1)[0]

    results.append({
        "query": test["query"],
        "expected": test["expected"],
        "predicted": prediction["sanskrit"],
        "score": prediction["score"]
    })

results_df = pd.DataFrame(results)
results_df.to_csv("retrieval_results.csv", index=False)

print("Results saved as retrieval_results.csv")

Results saved as retrieval_results.csv
